# Weibo DB quick look

This notebook inspects the local Weibo crawler database. It uses `data/weibo.duckdb` first, and can fall back to split JSON or legacy JSON if you point `DB_PATH` somewhere else.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import shutil

import duckdb
from IPython.display import display

DB_PATH = Path("data/weibo.duckdb")
SPLIT_JSON_DIR = Path("data/weibo")
LEGACY_JSON_PATH = Path("data/weibo.json")

WEIBO_COLLECTIONS = [
    "weibo_authors",
    "weibo_posts_raw",
    "weibo_comments",
]

TMP_DIR = None
CON = None
BACKEND = None

def connect_duckdb(path: Path):
    """Connect read-only; if another process holds the DB lock, inspect a temp snapshot."""
    global TMP_DIR
    try:
        return duckdb.connect(str(path), read_only=True), "duckdb"
    except Exception as exc:
        message = str(exc)
        if "Could not set lock" not in message and "Conflicting lock" not in message:
            raise
        TMP_DIR = TemporaryDirectory()
        snapshot = Path(TMP_DIR.name) / path.name
        shutil.copy2(path, snapshot)
        wal = path.with_suffix(path.suffix + ".wal")
        if wal.exists():
            shutil.copy2(wal, snapshot.with_suffix(snapshot.suffix + ".wal"))
        return duckdb.connect(str(snapshot), read_only=True), "duckdb snapshot (source DB was locked)"

def load_json_file(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def load_json_collection(collection: str):
    split_path = SPLIT_JSON_DIR / f"{collection}.json"
    if split_path.exists():
        payload = load_json_file(split_path)
        records = payload.get("records", {})
        return list(records.values()), "split-json"
    if LEGACY_JSON_PATH.exists():
        payload = load_json_file(LEGACY_JSON_PATH)
        records = payload.get("collections", {}).get(collection, {})
        return list(records.values()), "legacy-json"
    return [], "missing"

if DB_PATH.exists():
    CON, BACKEND = connect_duckdb(DB_PATH)
elif SPLIT_JSON_DIR.exists() or LEGACY_JSON_PATH.exists():
    BACKEND = "json"
else:
    raise FileNotFoundError("No Weibo database found. Expected data/weibo.duckdb, data/weibo/, or data/weibo.json.")

print(f"Backend: {BACKEND}")
print(f"DB path: {DB_PATH.resolve() if DB_PATH.exists() else 'not found'}")

def sql_rows(sql: str, params: list | None = None):
    if CON is None:
        raise RuntimeError("DuckDB connection is not available.")
    cursor = CON.execute(sql, params or [])
    columns = [column[0] for column in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]

## Collection Counts

In [ ]:
if CON is not None:
    display(sql_rows("""
        SELECT
            collection,
            count(*) AS rows,
            min(updated_at) AS first_updated_at,
            max(updated_at) AS last_updated_at
        FROM records
        GROUP BY collection
        ORDER BY collection
    """))
else:
    rows = []
    for collection in WEIBO_COLLECTIONS:
        records, backend = load_json_collection(collection)
        rows.append({"collection": collection, "backend": backend, "rows": len(records)})
    display(rows)

## Header / Key Coverage

These tables show the top-level fields found in each collection, how often they are populated, and the JSON types seen in those fields.

In [ ]:
def key_profile(records: list[dict], *, max_records: int | None = None):
    sample = records if max_records is None else records[:max_records]
    keys = sorted({key for record in sample if isinstance(record, dict) for key in record})
    rows = []
    total = len(sample)
    for key in keys:
        values = [record.get(key) for record in sample if isinstance(record, dict)]
        populated_values = [value for value in values if value not in (None, "", [], {})]
        type_counts = Counter(type(value).__name__ for value in populated_values)
        rows.append({
            "field": key,
            "non_empty": len(populated_values),
            "empty": total - len(populated_values),
            "coverage_pct": round((len(populated_values) / total * 100), 2) if total else 0,
            "types": ", ".join(f"{name}:{count}" for name, count in type_counts.most_common()),
        })
    return rows

def duckdb_records(collection: str, limit: int | None = None):
    sql = "SELECT data FROM records WHERE collection = ? ORDER BY id"
    params = [collection]
    if limit is not None:
        sql += " LIMIT ?"
        params.append(limit)
    return [json.loads(row[0]) for row in CON.execute(sql, params).fetchall()]

def records_for(collection: str, limit: int | None = None):
    if CON is not None:
        return duckdb_records(collection, limit=limit), BACKEND
    records, backend = load_json_collection(collection)
    return (records[:limit] if limit is not None else records), backend

for collection in WEIBO_COLLECTIONS:
    records, backend = records_for(collection)
    print(f"{collection} ({backend}) rows={len(records)}")
    display(key_profile(records))

## Header Rows / Sample Records

Nested dict/list fields and long strings are compacted so the first few rows stay readable.

In [ ]:
def compact_value(value, max_chars: int = 160):
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False, default=str)
    else:
        text = str(value) if value is not None else None
    if text is not None and len(text) > max_chars:
        return text[:max_chars] + "..."
    return text

def compact_rows(records: list[dict], *, rows: int = 5, max_chars: int = 160):
    return [
        {key: compact_value(value, max_chars=max_chars) for key, value in record.items()}
        for record in records[:rows]
        if isinstance(record, dict)
    ]

for collection in WEIBO_COLLECTIONS:
    records, _ = records_for(collection, limit=5)
    print(f"{collection}: first {len(records)} rows")
    display(compact_rows(records))

## Descriptive Stats

In [ ]:
if CON is not None:
    print("Post retrieval status")
    display(sql_rows("""
        SELECT
            json_extract_string(data, '$.status') AS status,
            count(*) AS posts
        FROM records
        WHERE collection = 'weibo_posts_raw'
        GROUP BY status
        ORDER BY posts DESC
    """))

    print("Post coverage")
    display(sql_rows("""
        SELECT
            count(*) AS posts,
            count(*) FILTER (WHERE nullif(json_extract_string(data, '$.url'), '') IS NOT NULL) AS posts_with_url,
            count(*) FILTER (WHERE nullif(json_extract_string(data, '$.content_html'), '') IS NOT NULL
                           OR nullif(json_extract_string(data, '$.html'), '') IS NOT NULL) AS posts_with_html,
            count(DISTINCT json_extract_string(data, '$.author_id')) AS authors_with_posts
        FROM records
        WHERE collection = 'weibo_posts_raw'
    """))

    print("Comments per post")
    display(sql_rows("""
        WITH comments AS (
            SELECT json_extract_string(data, '$.post_id') AS post_id
            FROM records
            WHERE collection = 'weibo_comments'
        )
        SELECT
            count(*) AS commented_posts,
            min(comment_count) AS min_comments,
            avg(comment_count) AS avg_comments,
            median(comment_count) AS median_comments,
            max(comment_count) AS max_comments
        FROM (
            SELECT post_id, count(*) AS comment_count
            FROM comments
            WHERE post_id IS NOT NULL AND post_id != ''
            GROUP BY post_id
        )
    """))

    print("Author numeric stats")
    display(sql_rows("""
        SELECT metric, count(*) AS n, min(value) AS min, avg(value) AS avg, median(value) AS median, max(value) AS max
        FROM (
            SELECT 'num_followers' AS metric, try_cast(json_extract_string(data, '$.num_followers') AS DOUBLE) AS value FROM records WHERE collection = 'weibo_authors'
            UNION ALL SELECT 'num_following', try_cast(json_extract_string(data, '$.num_following') AS DOUBLE) FROM records WHERE collection = 'weibo_authors'
            UNION ALL SELECT 'num_posts', try_cast(json_extract_string(data, '$.num_posts') AS DOUBLE) FROM records WHERE collection = 'weibo_authors'
            UNION ALL SELECT 'num_received_comments', try_cast(json_extract_string(data, '$.num_received_comments') AS DOUBLE) FROM records WHERE collection = 'weibo_authors'
            UNION ALL SELECT 'num_received_likes', try_cast(json_extract_string(data, '$.num_received_likes') AS DOUBLE) FROM records WHERE collection = 'weibo_authors'
            UNION ALL SELECT 'num_received_reposts', try_cast(json_extract_string(data, '$.num_received_reposts') AS DOUBLE) FROM records WHERE collection = 'weibo_authors'
        )
        WHERE value IS NOT NULL
        GROUP BY metric
        ORDER BY metric
    """))
else:
    authors, _ = records_for('weibo_authors')
    posts, _ = records_for('weibo_posts_raw')
    comments, _ = records_for('weibo_comments')
    print('Post retrieval status')
    display(Counter(post.get('status') for post in posts))
    print('Post coverage')
    display({
        'posts': len(posts),
        'posts_with_url': sum(bool(post.get('url')) for post in posts),
        'posts_with_html': sum(bool(post.get('content_html') or post.get('html')) for post in posts),
        'authors_with_posts': len({post.get('author_id') for post in posts if post.get('author_id')}),
    })
    print('Comments per post')
    per_post = Counter(comment.get('post_id') for comment in comments if comment.get('post_id'))
    values = sorted(per_post.values())
    display({
        'commented_posts': len(values),
        'min_comments': values[0] if values else None,
        'avg_comments': sum(values) / len(values) if values else None,
        'median_comments': values[len(values) // 2] if values else None,
        'max_comments': values[-1] if values else None,
    })

## Top Rows To Inspect

In [ ]:
if CON is not None:
    print("Most commented posts in local comment table")
    display(sql_rows("""
        SELECT
            json_extract_string(data, '$.post_id') AS post_id,
            count(*) AS comments
        FROM records
        WHERE collection = 'weibo_comments'
        GROUP BY post_id
        HAVING post_id IS NOT NULL AND post_id != ''
        ORDER BY comments DESC, post_id
        LIMIT 20
    """))

    print("Authors with most stored posts")
    display(sql_rows("""
        SELECT
            json_extract_string(data, '$.author_id') AS author_id,
            count(*) AS posts,
            count(*) FILTER (WHERE json_extract_string(data, '$.status') = 'RETRIEVED') AS retrieved_posts
        FROM records
        WHERE collection = 'weibo_posts_raw'
        GROUP BY author_id
        HAVING author_id IS NOT NULL AND author_id != ''
        ORDER BY posts DESC, author_id
        LIMIT 20
    """))
else:
    posts, _ = records_for('weibo_posts_raw')
    comments, _ = records_for('weibo_comments')
    display(Counter(comment.get('post_id') for comment in comments if comment.get('post_id')).most_common(20))
    display(Counter(post.get('author_id') for post in posts if post.get('author_id')).most_common(20))

In [ ]:
# Close the connection when you are done with the notebook.
if CON is not None:
    CON.close()
if TMP_DIR is not None:
    TMP_DIR.cleanup()